# Módulo 10 · Lista de Exercícios — e o fechamento do manual

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Este é o último módulo. Ao final desta lista, o Atlas responde a pergunta que a Aurora fazia lá no Módulo 01 — *"quanto vendemos por cidade?"* — mas agora com dado de ontem à noite, calculado sozinho, conferido, versionado e sem tocar no banco que atende o cliente.

## Como usar

| | |
|---|---|
| 🟢 **Aquecimento** | 1–12 · uma ideia por exercício |
| 🟡 **Construção** | 13–30 · combinam conceitos |
| 🔴 **Integração** | 31–44 · perto de produção |
| 🏗️ **Projeto final** | O pipeline de dados do Atlas |

**Regras de casa:**

1. Todo número que você afirmar, meça.
2. Todo 🔴 tem uma armadilha que produz **número errado sem erro aparente**. São as piores.
3. Quebre o pipeline de propósito. Um portão de qualidade que você nunca viu reprovar não é um portão.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 10
# ═══════════════════════════════════════════════════════════════
import json
import os
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import time
import warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("pandas", "pandas"), ("numpy", "numpy"),
               ("pyarrow", "pyarrow"), ("polars", "polars"),
               ("duckdb", "duckdb")]:
    _garantir(_p, _m)

import numpy as np
import pandas as pd

TEM_POLARS = _garantir("polars", "polars")
TEM_DUCKDB = _garantir("duckdb", "duckdb")
TEM_ARROW = _garantir("pyarrow", "pyarrow")

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"pandas {pd.__version__} · numpy {np.__version__}")
if TEM_POLARS:
    import polars as pl
    print(f"polars {pl.__version__}")
if TEM_DUCKDB:
    import duckdb
    print(f"duckdb {duckdb.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Dados da Aurora — gerados de forma REPRODUTÍVEL
# ═══════════════════════════════════════════════════════════════
SEMENTE = 20260813
rng = np.random.default_rng(SEMENTE)
random.seed(SEMENTE)

CIDADES = ["Campinas", "São Paulo", "Valinhos", "Sumaré",
           "Indaiatuba", "Jundiaí", "Hortolândia"]
CANAIS = ["site", "app", "marketplace"]
CATEGORIAS = {
    "NB": ("Notebooks", 1800, 4200),
    "MO": ("Monitores", 700, 2200),
    "PE": ("Periféricos", 40, 400),
    "AR": ("Armazenamento", 180, 900),
}


def gerar_produtos(n: int = 60) -> pd.DataFrame:
    linhas = []
    for i in range(n):
        prefixo = list(CATEGORIAS)[i % len(CATEGORIAS)]
        categoria, minimo, maximo = CATEGORIAS[prefixo]
        preco = round(float(rng.uniform(minimo, maximo)), 2)
        linhas.append({
            "sku": f"{prefixo}-{1000 + i}",
            "nome": f"{categoria[:-1]} modelo {i:03d}",
            "categoria": categoria,
            "preco": preco,
            "custo": round(preco * float(rng.uniform(0.55, 0.85)), 2),
            "estoque": int(rng.integers(0, 200)),
        })
    return pd.DataFrame(linhas)


def gerar_vendas(n: int = 50_000, dias: int = 180,
                 produtos: pd.DataFrame | None = None) -> pd.DataFrame:
    """Vendas sintéticas com sazonalidade e um pouco de sujeira."""
    produtos = gerar_produtos() if produtos is None else produtos
    fim = datetime(2026, 8, 1, tzinfo=timezone.utc)
    inicio = fim - timedelta(days=dias)

    idx = rng.integers(0, len(produtos), n)
    escolhidos = produtos.iloc[idx].reset_index(drop=True)

    # 📈 Sazonalidade: mais vendas no fim de semana e no fim do mês
    deslocamento = rng.integers(0, dias, n)
    datas = pd.to_datetime(inicio) + pd.to_timedelta(deslocamento, unit="D")
    datas = datas + pd.to_timedelta(rng.integers(0, 86400, n), unit="s")

    return pd.DataFrame({
        "pedido_id": 100_000 + np.arange(n),
        "data": datas,
        "sku": escolhidos["sku"],
        "categoria": escolhidos["categoria"],
        "cidade": rng.choice(CIDADES, n, p=[.28, .22, .12, .12, .11, .09, .06]),
        "canal": rng.choice(CANAIS, n, p=[.55, .30, .15]),
        "quantidade": rng.integers(1, 6, n),
        "preco_unitario": escolhidos["preco"],
        "custo_unitario": escolhidos["custo"],
        "status": rng.choice(["pago", "pendente", "cancelado"], n, p=[.82, .10, .08]),
        "frete": np.round(rng.uniform(0, 45, n), 2),
    })


# ═══════════════════════════════════════════════════════════════
#  Medição
# ═══════════════════════════════════════════════════════════════

def cronometrar(funcao, repeticoes: int = 1):
    """Devolve (resultado, milissegundos_medios)."""
    inicio = time.perf_counter()
    resultado = None
    for _ in range(repeticoes):
        resultado = funcao()
    return resultado, (time.perf_counter() - inicio) * 1000 / repeticoes


def comparar(casos: list[tuple[str, callable]], repeticoes: int = 1,
             rotulo: str = "abordagem"):
    """Mede várias abordagens e mostra o ganho relativo."""
    medidos = []
    for nome, funcao in casos:
        _, ms = cronometrar(funcao, repeticoes)
        medidos.append((nome, ms))
    melhor = min(m for _, m in medidos)
    largura = max(len(n) for n, _ in medidos) + 2
    print(f"{rotulo:<{largura}}{'tempo':>12}   {'vs melhor':>10}")
    print("─" * (largura + 26))
    for nome, ms in medidos:
        barra = "█" * max(1, int(ms / melhor))
        print(f"{nome:<{largura}}{ms:>9.1f} ms   {ms / melhor:>8.1f}×  {barra[:26]}")
    return medidos


def tamanho(n: int) -> str:
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:,.1f} {unidade}" if unidade != "B" else f"{n:,} B"
        n /= 1024
    return ""


def memoria(df: pd.DataFrame) -> int:
    return int(df.memory_usage(deep=True).sum())


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("\n✅ `gerar_vendas()`, `comparar()`, `memoria()`, `tabela()` prontos")
print(f"   semente fixa ({SEMENTE}) — os números são reprodutíveis")

---

# 🟢 Aquecimento (1–12)

### 1 · OLTP vs OLAP

Escreva três perguntas de cada tipo sobre o Atlas. Meça a diferença entre uma busca por chave e uma agregação completa.

In [ ]:
# 1

### 2 · Colunar

Explique por que o formato colunar comprime melhor. Prove com uma coluna de baixa cardinalidade do Atlas.

In [ ]:
# 2

### 3 · As camadas

Monte bronze, prata, ouro e quarentena. Explique o papel de cada uma numa frase.

In [ ]:
# 3

### 4 · `loc` vs `iloc`

Mostre a diferença nas fatias e explique por que uma inclui o fim.

In [ ]:
# 4

### 5 · 🔴 A alteração que não acontece

Reproduza a atribuição que se perde num recorte. Corrija das duas formas.

In [ ]:
# 5

### 6 · 🔴 Nulos

Compare `mean()`, `sum()/count()` e `sum()/len()` numa coluna com nulos. Explique de onde vem a diferença.

In [ ]:
# 6

### 7 · 🔴 `groupby` e nulos

Mostre a soma dos grupos não batendo com o total. Corrija com `dropna=False`.

In [ ]:
# 7

### 8 · Vetorização

Compare `iterrows`, `apply(axis=1)` e vetorizado em 100 mil linhas.

In [ ]:
# 8

### 9 · 🔴 `merge` que multiplica

Reproduza a duplicação com chave repetida à direita. Mostre `validate=` pegando.

In [ ]:
# 9

### 10 · dtypes

Otimize os tipos de um DataFrame e meça a redução. Justifique por que dinheiro fica em `float64`.

In [ ]:
# 10

### 11 · Parquet

Salve o mesmo dado em CSV e Parquet. Compare tamanho, tempo e leitura de 3 colunas.

In [ ]:
# 11

### 12 · CSV brasileiro

Leia um CSV com `;`, `latin-1`, vírgula decimal e data `dd/mm/aaaa`. Mostre o que quebra sem cada ajuste.

In [ ]:
# 12

---

# 🟡 Construção (13–30)

### 13 · `transform` vs `agg`

Calcule a participação de cada venda no total do seu grupo. Explique por que `agg` não serve.

In [ ]:
# 13

### 14 · Séries temporais

Reamostre por dia, semana e mês. Calcule a média móvel de 7 dias com `min_periods`.

In [ ]:
# 14

### 15 · 🔴 Fuso horário

Mostre que agrupar por dia em UTC dá resultado diferente de agrupar em horário local.

In [ ]:
# 15

### 16 · Relatório auditável

Escreva uma função que devolva o resultado **e** os metadados das decisões (filtros, fuso, tratamento de nulo).

In [ ]:
# 16

### 17 · Leitura em pedaços

Agregue um arquivo grande com `chunksize`. Prove que dá o mesmo resultado.

In [ ]:
# 17

### 18 · 🔴 Agregações que não se decompõem

Explique por que mediana e contagem de distintos não funcionam em pedaços.

In [ ]:
# 18

### 19 · Particionamento

Particione um Parquet por ano e mês. Meça o ganho ao filtrar um mês.

In [ ]:
# 19

### 20 · 🎯 Polars preguiçoso

Escreva uma consulta com `scan_parquet` e leia o `explain()`. Aponte a projeção e o filtro empurrados.

In [ ]:
# 20

### 21 · DuckDB

Faça a mesma agregação em SQL, direto sobre o Parquet. Compare com o Pandas.

In [ ]:
# 21

### 22 · 🔴 Excel do fornecedor

Leia uma planilha com cabeçalho na linha 7, linha de total e código com zero à esquerda. Mostre o faturamento errado se a linha de total entrar.

In [ ]:
# 22

### 23 · Conferência de formato

Escreva um leitor que falhe cedo se a planilha mudar de layout.

In [ ]:
# 23

### 24 · JSON aninhado

Use `json_normalize` com `record_path` e `meta`. Explique o que se perde sem `meta`.

In [ ]:
# 24

### 25 · 🎯 Ingestão incremental

Implemente a marca d'água. Rode três vezes e mostre a terceira lendo só o que chegou.

In [ ]:
# 25

### 26 · 🔴 As armadilhas da marca d'água

Explique as três (data retroativa, relógios diferentes, exclusão) e proponha defesa para cada uma.

In [ ]:
# 26

### 27 · 🔴 Idempotência da carga

Mostre `if_exists='append'` duplicando. Corrija com UPSERT e regra de desempate.

In [ ]:
# 27

### 28 · Contrato Pydantic

Escreva o contrato dos seus dados, com normalização em `mode="before"`.

In [ ]:
# 28

### 29 · 🔴 Quarentena

Implemente. Prove que uma linha ruim não derruba o lote nem desaparece.

In [ ]:
# 29

### 30 · Scraping educado

Extraia dados de um HTML local, consulte um `robots.txt` e respeite o `Crawl-delay`.

In [ ]:
# 30

---

# 🔴 Integração (31–44)

### 31 · 🎯 As seis verificações

Implemente completude, unicidade, não-nulos, faixa, coerência e volume. Provoque cada falha.

In [ ]:
# 31

### 32 · 🔑 Volume vs histórico

Explique por que essa verificação pega o que as outras cinco não pegam. Demonstre com um lote truncado.

In [ ]:
# 32

### 33 · Erro vs aviso

Classifique dez verificações em "aborta" e "avisa". Justifique as que não abortam.

In [ ]:
# 33

### 34 · 🔴 O portão

Faça o pipeline **não publicar** o ouro quando a prata reprova. Argumente por que dado velho e certo é melhor.

In [ ]:
# 34

### 35 · Conciliação

Concilie prata e ouro com `assert`. Quebre de propósito e veja falhar.

In [ ]:
# 35

### 36 · 🔑 Reprocessamento

Reprocesse um período a partir do bronze. Rode duas vezes e prove a idempotência.

In [ ]:
# 36

### 37 · 🔴 Trava distribuída

Implemente com `SET NX EX`. Prove com 5 threads que só uma executa. Explique por que `threading.Lock` não serve.

In [ ]:
# 37

### 38 · Fila confiável

Implemente com `BRPOPLPUSH` e confirmação. Mostre a mensagem sobrevivendo à morte do worker.

In [ ]:
# 38

### 39 · 🔴 Consumo idempotente

Entregue a mesma tarefa três vezes e processe uma. Relacione com o webhook do M07.

In [ ]:
# 39

### 40 · 🎯 Executor de DAG

Escreva um com ordenação topológica, retry e propagação de falha. Detecte ciclos.

In [ ]:
# 40

### 41 · 🔴 Falha propagada

Faça uma etapa falhar e prove que as dependentes são **puladas**, não executadas com dado velho.

In [ ]:
# 41

### 42 · 🎯 Data lógica

Reescreva uma tarefa que usa `date.today()` para receber a data. Explique o que isso habilita.

In [ ]:
# 42

### 43 · Backfill

Reprocesse 5 dias. Mostre um arquivo por dia e explique o que aconteceria com `date.today()`.

In [ ]:
# 43

### 44 · 🔴 Dicionário de métricas

Escreva a ficha de três métricas do Atlas, incluindo o que elas **não** incluem e qual fuso usam.

In [ ]:
# 44

---

# 🏗️ Projeto final — O pipeline de dados do Atlas

## O contexto

Voltamos ao começo:

> *"Ninguém sabe quanto vendemos por cidade. Toda segunda alguém passa a tarde somando planilha à mão e o número nunca bate."*
> — Diretora Comercial, **Módulo 01**

No M01 você resolveu isso com um script Python lendo um CSV. Funcionava — para um CSV, na sua máquina, rodado por você.

Agora a resposta é outra:

> A pergunta é respondida por uma tabela que foi calculada às 3h, a partir de dados extraídos incrementalmente do Postgres, validados contra um contrato, com as linhas ruins em quarentena, conferidos por seis verificações de qualidade, e publicados apenas se passaram. **Sem tocar no banco que atende o cliente.**

## O que entregar

```
projeto_Atlas/
├── src/atlas/dados/
│   ├── __init__.py
│   ├── extracao.py        ← banco, CSV, API — com marca d'água
│   ├── contratos.py       ← os modelos Pydantic
│   ├── transformacao.py   ← bronze → prata → ouro
│   ├── qualidade.py       ← as seis verificações
│   └── orquestracao.py    ← o DAG e o executor
├── dados/lago/
│   ├── bronze/  prata/  ouro/  quarentena/  estado/
├── scripts/
│   └── rodar_pipeline.py  ← com trava e data como parâmetro
└── docs/
    ├── PIPELINE.md        ← as decisões
    └── METRICAS.md        ← 🔑 o dicionário
```

## Requisitos obrigatórios

| # | Requisito | Pronto quando |
|---|-----------|---------------|
| 1 | Extração incremental | A 2ª execução lê pouco |
| 2 | 🔑 Bronze imutável | Com manifesto e hash |
| 3 | Contrato Pydantic | Normaliza e valida |
| 4 | 🔴 Quarentena | Linha ruim não derruba nem some |
| 5 | Deduplicação por chave | Com regra de desempate |
| 6 | 🎯 As seis verificações | Cada uma provocada e vista falhar |
| 7 | 🔴 Portão de qualidade | Reprova → não publica o ouro |
| 8 | Conciliação | `assert` entre camadas |
| 9 | 🔴 Idempotência | Rodar 2× dá o mesmo resultado |
| 10 | Reprocessamento | Backfill de um período |
| 11 | 🔴 Trava distribuída | Duas execuções não se sobrepõem |
| 12 | 🎯 Data como parâmetro | Nenhum `date.today()` na lógica |
| 13 | Dicionário de métricas | Com o que elas não incluem |

> 📋 **O roteiro está em `projeto_Atlas/ROTEIRO_M10.md`.**

## 🧪 Bateria de aceitação

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Aponte para o SEU projeto
# ═══════════════════════════════════════════════════════════════
PROJETO = None          # ← troque pelo caminho do seu projeto_Atlas

RESULTADOS = []


def checar(nome: str, condicao: bool, detalhe: str = "") -> bool:
    RESULTADOS.append((nome, bool(condicao)))
    print(f"   {'✅' if condicao else '🔴'} {nome}{('  — ' + detalhe) if detalhe else ''}")
    return bool(condicao)


def placar():
    passou = sum(1 for _, ok in RESULTADOS if ok)
    print(f"\n{'═' * 56}\n  {passou}/{len(RESULTADOS)} verificações passaram")
    if RESULTADOS and passou == len(RESULTADOS):
        print("  🎉 pipeline de dados aprovado.")
    else:
        for nome, ok in RESULTADOS:
            if not ok:
                print(f"  🔴 pendente: {nome}")
    print("═" * 56)


print("⏸️  defina `PROJETO` acima para rodar a bateria."
      if PROJETO is None else f"▶️  auditando {PROJETO}")

In [ ]:
# ── Bateria 1: a estrutura do lago ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    lago = PROJETO / "dados" / "lago"
    checar("existe dados/lago/", lago.is_dir())

    for camada in ("bronze", "prata", "ouro", "quarentena"):
        checar(f"existe a camada {camada}", (lago / camada).is_dir())

    if (lago / "bronze").is_dir():
        particoes = [p for p in (lago / "bronze").rglob("*") if p.is_dir()
                     and "=" in p.name]
        checar("🔑 bronze é particionado", bool(particoes),
               f"{len(particoes)} partição(ões)")
        manifestos = list((lago / "bronze").rglob("_manifesto.json"))
        checar("bronze tem manifesto", bool(manifestos),
               f"{len(manifestos)} manifesto(s)")

    for modulo in ["extracao.py", "contratos.py", "transformacao.py",
                   "qualidade.py"]:
        checar(f"existe dados/{modulo}",
               (PROJETO / "src" / "atlas" / "dados" / modulo).exists())

    placar()

In [ ]:
# ── Bateria 2: 🎯 as verificações de qualidade existem ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    arquivo = PROJETO / "src" / "atlas" / "dados" / "qualidade.py"
    if not arquivo.exists():
        print("   ⏸️  crie src/atlas/dados/qualidade.py")
    else:
        texto = arquivo.read_text(encoding="utf-8").lower()
        verificacoes = {
            "completude (lote não vazio)": ["vazio", "len(", "completude"],
            "unicidade da chave":          ["duplicat", "unico", "único", "unique"],
            "colunas obrigatórias":        ["isna", "nulo", "obrigat"],
            "faixa de valores":            ["> 0", ">0", "faixa", "maximo", "máximo"],
            "coerência entre colunas":     ["coeren", "coerên", "margem", "<="],
            "🔑 volume vs histórico":      ["volume", "histor", "histór"],
        }
        for nome, palavras in verificacoes.items():
            checar(f"há verificação de {nome}",
                   any(p in texto for p in palavras))

        checar("distingue erro de aviso",
               "aviso" in texto and "erro" in texto,
               "nem toda falha deve abortar")

        placar()
        print("\n   ⚠️  esta bateria procura PALAVRAS no arquivo — é um verificador fraco.")
        print("      Um nome de parâmetro pode fazê-la passar sem que a verificação exista.")
        print("      Só a Bateria 3, que EXECUTA o pipeline, prova comportamento.")

In [ ]:
# ── Bateria 3: 🔴 o pipeline é idempotente ──
#
# 🎯 A bateria que mais reprova — e a que mais importa.
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    script = PROJETO / "scripts" / "rodar_pipeline.py"
    checar("existe scripts/rodar_pipeline.py", script.exists())

    if script.exists():
        texto = script.read_text(encoding="utf-8")

        # 🔴 nenhum date.today() na lógica
        usa_hoje = bool(re.search(r"date\.today\(\)|datetime\.now\(\)\.date\(\)", texto))
        tem_argumento = "argv" in texto or "argparse" in texto
        checar("🎯 a data vem de fora (argumento)", tem_argumento,
               "sem isso, backfill é impossível")
        checar("🔴 a data lógica não é 'hoje' cega",
               tem_argumento or not usa_hoje,
               "date.today() na lógica quebra o reprocessamento")

        checar("🔴 usa trava para não sobrepor",
               any(p in texto.lower() for p in ("trava", "lock", "nx=true")))

        checar("devolve código de saída",
               "sys.exit" in texto or "return 0" in texto,
               "o agendador precisa saber se falhou")

    # A prova de fogo: rodar duas vezes
    lago = PROJETO / "dados" / "lago"
    if lago.is_dir() and script.exists():
        def impressao_digital() -> dict:
            """Hash de cada arquivo do ouro."""
            import hashlib
            return {p.name: hashlib.sha256(p.read_bytes()).hexdigest()[:12]
                    for p in sorted((lago / "ouro").glob("*.parquet"))}

        print("\n   ── rodando o pipeline duas vezes ──")
        env = dict(os.environ)
        r1 = subprocess.run([sys.executable, str(script), "2026-08-13"],
                            cwd=PROJETO, capture_output=True, text=True, env=env)
        digital1 = impressao_digital()
        r2 = subprocess.run([sys.executable, str(script), "2026-08-13"],
                            cwd=PROJETO, capture_output=True, text=True, env=env)
        digital2 = impressao_digital()

        checar("o pipeline roda sem erro", r1.returncode == 0,
               (r1.stderr or r1.stdout).strip().splitlines()[-1][:60]
               if r1.returncode else "")

        # 🔑 Sem esta linha, a verificação abaixo APROVA um pipeline que
        #    não produziu nada: {} == {} é True. Um lote vazio comparado
        #    com outro lote vazio é "idempotente" da forma mais inútil
        #    possível. Leia a nota no fim desta bateria.
        checar("o pipeline produziu ouro", len(digital1) > 0,
               "sem isto, a verificação seguinte compara vazio com vazio")

        checar("🔴 rodar 2× produz o MESMO ouro",
               bool(digital1) and digital1 == digital2,
               f"{len(digital1)} arquivo(s) comparados")

    placar()
    print("\n   💭 Esta bateria tinha um bug, e o bug virou a lição:")
    print("      a comparação de hashes APROVAVA um pipeline que não")
    print("      gerava arquivo nenhum — porque {} == {} é True.")
    print("      Toda verificação que compara coleções precisa antes")
    print("      exigir que a coleção NÃO esteja vazia. É o mesmo erro")
    print("      de `all([])`, que também devolve True.")

In [ ]:
# ── Bateria 4: 🔑 documentação das decisões ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    metricas = PROJETO / "docs" / "METRICAS.md"
    checar("existe docs/METRICAS.md", metricas.exists(),
           "o dicionário que encerra a discussão sobre 'qual número é o certo'")

    if metricas.exists():
        texto = metricas.read_text(encoding="utf-8").lower()
        for tema, palavras in {
            "definição do cálculo":   ["definição", "definicao", "="],
            "🔑 o que NÃO inclui":    ["não inclui", "nao inclui", "não entra", "exclui"],
            "filtros aplicados":      ["filtro", "status", "apenas", "somente"],
            "fuso horário":           ["fuso", "utc", "brasília", "brasilia"],
            "origem do dado":         ["fonte", "origem", "lago/"],
        }.items():
            checar(f"documenta {tema}", any(p in texto for p in palavras))

    checar("existe docs/PIPELINE.md", (PROJETO / "docs" / "PIPELINE.md").exists())

    placar()

> 🎯 **A bateria 3 é a que separa um script de um pipeline.**
>
> Ela roda o seu pipeline **duas vezes com a mesma data** e compara o hash de cada arquivo da camada ouro. Se os hashes diferirem, alguma etapa não é idempotente — e um dia alguém vai rodar duas vezes por engano.
>
> 💭 **É o mesmo teste que atravessa o manual inteiro, em três formas:**
>
> | Módulo | A pergunta |
> |--------|-----------|
> | M07 | O teste falha quando o código está errado? |
> | M08 | O verificador reprova o Dockerfile ruim? |
> | M09 | O portão do CI barra o segredo plantado? |
> | **M10** | **Rodar duas vezes muda o resultado?** |
>
> **Verificar que a verificação funciona é o hábito que este manual mais tentou construir.**

---

## 🎓 Autoavaliação

| # | Consigo… | ✅ |
|---|----------|---|
| 1 | Explicar por que analítica no banco de produção derruba o site | |
| 2 | Justificar o formato colunar para agregação | |
| 3 | Explicar o papel de cada camada e por que o bronze é imutável | |
| 4 | Distinguir fato de dimensão | |
| 5 | 🔴 Explicar por que o fato guarda o preço do momento da venda | |
| 6 | Usar `loc`/`iloc` sem hesitar | |
| 7 | 🔴 Corrigir a atribuição que se perde num recorte | |
| 8 | 🔴 Explicar por que `mean()` difere de `sum()/len()` | |
| 9 | 🔴 Saber que `groupby` descarta chave nula | |
| 10 | Vetorizar em vez de `apply(axis=1)` | |
| 11 | 🔴 Usar `validate=` contra a multiplicação de linhas | |
| 12 | Explicar por que agrupar por dia UTC ≠ dia local | |
| 13 | Otimizar dtypes e saber por que dinheiro fica em float64 | |
| 14 | Explicar as três vantagens do Parquet | |
| 15 | 🎯 Ler um plano de consulta do Polars | |
| 16 | Consultar Parquet com SQL no DuckDB | |
| 17 | Ler CSV brasileiro e Excel com cabeçalho torto | |
| 18 | 🔴 Explicar por que identificador é sempre texto | |
| 19 | 🎯 Implementar ingestão incremental com marca d'água | |
| 20 | 🔴 Citar as três armadilhas da marca d'água | |
| 21 | 🔴 Escrever uma carga idempotente com UPSERT | |
| 22 | Escrever um contrato Pydantic com normalização | |
| 23 | 🔴 Implementar quarentena sem derrubar nem descartar | |
| 24 | 🎯 Implementar as seis verificações de qualidade | |
| 25 | 🔑 Explicar por que a verificação de volume é especial | |
| 26 | Distinguir falha que aborta de falha que avisa | |
| 27 | 🔴 Justificar não publicar o ouro quando a prata reprova | |
| 28 | Reprocessar um período a partir do bronze | |
| 29 | 🔴 Implementar trava distribuída e explicar por que `Lock` não serve | |
| 30 | 🎯 Escrever um executor de DAG com retry e propagação | |
| 31 | 🎯 Explicar por que `date.today()` impede o backfill | |
| 32 | 🔴 Explicar o que o `cron` não sabe | |
| 33 | Argumentar que a Aurora não precisa de Kafka hoje | |
| 34 | Escrever a ficha de uma métrica, com o que ela não inclui | |

**Menos de 27?** Volte às aulas antes de considerar o módulo concluído.

---

# 🎓 O fim do manual — e o que você construiu

In [ ]:
jornada = [
    ["M01", "Python",      "CLI de relatórios sobre CSV"],
    ["M02", "Git",         "histórico, `.gitignore`, automações"],
    ["M03", "SQL",         "schema relacional, índices, transações"],
    ["M04", "OOP",         "camadas, tipos, logging estruturado"],
    ["M05", "Persistência", "PostgreSQL + MongoDB, migrações"],
    ["M06", "API",         "FastAPI autenticada e documentada"],
    ["M07", "Integrações", "resiliência, webhooks, testes"],
    ["M08", "Containers",  "imagem, compose, auditoria"],
    ["M09", "Deploy",      "CI/CD, proxy, monitoramento"],
    ["M10", "Dados",       "pipeline ETL com qualidade"],
]
tabela(["#", "TEMA", "O QUE VIROU O ATLAS"], jornada, [6, 14, 44])

print("""
🎯 A MESMA PERGUNTA, DEZ MÓDULOS DEPOIS

   M01:  python main.py relatorio --cidade
         lê um CSV · roda na sua máquina · você executa

   M10:  a tabela `ouro/faturamento_mensal.parquet`
         · extraída incrementalmente do Postgres
         · validada contra um contrato
         · com as linhas ruins em quarentena
         · conferida por seis verificações
         · publicada só se passou
         · calculada às 3h, sozinha
         · sem tocar no banco que atende o cliente
         · e reprocessável se você achar um bug
""")

In [ ]:
print("""
💭 AS IDEIAS QUE ATRAVESSARAM O MANUAL INTEIRO

   1. FALHE CEDO E BARATO
      validação na fronteira (M06) · lint antes do teste (M09)
      · conferir o formato antes de processar (M10)

   2. TORNE A REPETIÇÃO SEGURA
      migração idempotente (M03) · Idempotency-Key (M07)
      · bronze que substitui (M10) · UPSERT (M10)

   3. O QUE NÃO DEVE EXISTIR NUNCA PODE ENTRAR
      segredo fora do código (M06) · fora da imagem (M08)
      · fora do log e do CI (M09)

   4. DADO RUIM NÃO SOME NEM DERRUBA
      exceções de domínio (M01) · quarentena (M10)

   5. GUARDE A ENTRADA PARA PODER RECALCULAR
      Git (M02) · camada bronze (M10)

   6. VERIFIQUE QUE A VERIFICAÇÃO FUNCIONA
      🎯 o hábito mais importante de todos
      teste que falha antes de corrigir (M07)
      · verificador que reprova o Dockerfile ruim (M08)
      · portão de CI que barra o segredo plantado (M09)
      · pipeline que dá o mesmo resultado 2× (M10)

   7. DECISÃO DE NEGÓCIO SE DOCUMENTA, NÃO SE ADIVINHA
      docs/ em todos os módulos · dicionário de métricas (M10)
""")

In [ ]:
print("""
🧭 PARA ONDE IR AGORA

   O manual acaba aqui. O Atlas, não.

   ── Termine o projeto ──────────────────────────────────────
   Se você pulou algum `ROTEIRO_MXX.md`, ele é o próximo passo.
   O manual ensinou as peças; o Atlas é onde você descobre que
   sabe montá-las.

   ── Os três livros que continuam esta conversa ─────────────
   · Designing Data-Intensive Applications — Martin Kleppmann
   · Release It! — Michael Nygard          (M07 e M09)
   · Fundamentals of Data Engineering — Reis & Housley (M10)

   ── Os temas que ficaram de fora ───────────────────────────
   · testes de propriedade (Hypothesis)
   · dbt, para transformação declarativa
   · Delta/Iceberg, quando o lago precisar de transação
   · observabilidade distribuída (OpenTelemetry)
   · segurança ofensiva — leia o OWASP inteiro

   ── E o conselho que vale mais que a lista ─────────────────
   🎯 Escolha um problema REAL — seu, do seu trabalho, de um
      amigo — e resolva do começo ao fim.

      Um sistema que alguém usa ensina em três meses o que
      nenhum curso ensina em um ano. Este manual foi escrito
      para te dar as ferramentas; o que faz a diferença é ter
      um usuário esperando.
""")

In [ ]:
print(f"""
{'═' * 62}

   Você começou somando um CSV com um `for`.
   Terminou com um sistema que se testa, se publica e se
   recupera sozinho.

   A diferença entre os dois não foi a quantidade de
   ferramentas — foi a quantidade de perguntas que você
   aprendeu a fazer antes de escrever a primeira linha.

   Boa sorte, e bom código.

{'═' * 62}
""")

---

### 📋 Onde encontrar cada coisa

| Preciso de… | Aula |
|-------------|------|
| Por que não rodar analítica em produção | 10_01 |
| Camadas bronze/prata/ouro | 10_01 e 10_05 |
| `loc` vs `iloc`, nulos, `merge` | 10_02 |
| Vetorização e `groupby` | 10_02 |
| dtypes, memória, Parquet | 10_03 |
| Polars preguiçoso e DuckDB | 10_03 |
| CSV brasileiro e Excel torto | 10_04 |
| Ingestão incremental e UPSERT | 10_04 |
| Scraping e ética | 10_04 |
| Contratos, quarentena, qualidade | 10_05 |
| Reprocessamento e linhagem | 10_05 |
| Filas, trava distribuída, Celery | 10_06 |
| DAG, backfill, Airflow | 10_06 |

**E os módulos anteriores continuam valendo.** A cola de referência de cada aula foi escrita para ser consultada depois — é para isso que ela existe.